# Model comparison
Single-omics models and final comparator methods are evaluated using the same held-out outer folds as the proposed model.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_DIR))
from src import config
from src.comparison import FINAL_MODELS, aggregate as aggregate_comparator, run_outer_fold as run_comparator
from src.single_omics import MODALITIES, run_outer_fold as run_single_omics

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SINGLE_RESULT_DIR = config.OUTPUT_DIR / 'single_omics'
COMPARATOR_RESULT_DIR = config.OUTPUT_DIR / 'comparators'
RUN_SINGLE_OMICS = False
RUN_COMPARATORS = False
print('Device:', DEVICE)
print('Single-omics:', MODALITIES)
print('Comparators:', FINAL_MODELS)

In [ ]:
processed = torch.load(config.PROCESSED_DATA_PATH, map_location='cpu', weights_only=False)
print('Participants:', len(processed['participant_manifest']))

In [ ]:
if RUN_SINGLE_OMICS:
    for modality in MODALITIES:
        for outer_fold in range(1, config.OUTER_SPLITS + 1):
            row = run_single_omics(processed, modality, outer_fold, config, DEVICE, SINGLE_RESULT_DIR)
            print(f"[Completed] {modality} fold {outer_fold} | ACC={row['accuracy']:.4f} | ARI={row['ari']:.4f}")
else:
    print('Single-omics full run disabled.')

In [ ]:
if RUN_COMPARATORS:
    for model_name in FINAL_MODELS:
        for outer_fold in range(1, config.OUTER_SPLITS + 1):
            row = run_comparator(processed, model_name, outer_fold, config, COMPARATOR_RESULT_DIR)
            print(f"[Completed] {model_name} fold {outer_fold} | ACC={row['accuracy']:.4f} | ARI={row['ari']:.4f}")
        aggregate_comparator(COMPARATOR_RESULT_DIR, model_name)
else:
    print('Comparator full run disabled.')